
# SKIPGRAM reutiliza funciones de CBOW

Este repo tiene dos carpetas de notebooks, `notebooks_CBOW/` y `notebooks_SKIPGRAM/`,
y eso puede dar la impresión de dos proyectos. **No lo son**: comparten el mismo
`src/`, los mismos `configs/`, el mismo corpus y el mismo vocabulario. Las carpetas
separan *lo que se cuenta*, no *el código que lo ejecuta*.

Este notebook explica por qué se decidió así y lo demuestra con el propio código.

## Razón 1 — Ahorrar código (y bugs)

Copiar el proyecto habría duplicado 7 de los 9 módulos de `src/`:

| Módulo | ¿Cambia entre CBOW y skip-gram? |
|---|---|
| `corpus.py`, `vocabulary.py`, `config.py` | No |
| `train.py`, `evaluate.py`, `export.py`, `compare.py` | No |
| `dataset.py` | Sí, **una función** (`generate_skipgram_pairs`) |
| `model.py` | Sí, **un `forward`** (`SkipGramModel`) |

Cada módulo copiado es un lugar más donde arreglar el mismo bug dos veces, y donde
tarde o temprano una de las copias queda desactualizada.

## Razón 2 — La comparación exige el mismo espacio de índices

Para que CBOW y skip-gram sean comparables tienen que entrenar sobre **el mismo
vocabulario**, es decir, la misma correspondencia palabra ↔ índice. Eso lo resuelve
la llave `vocab_from` del YAML, que solo funciona dentro de un mismo proyecto. Con dos
carpetas habría que copiar `vocab.json` a mano en cada refresco y confiar en que nadie
se olvide.

## Razón 3 — La diferencia real es pequeña y está localizada

Está en **los pares**, no en el modelo ni en el entrenamiento. Sobre la misma ventana:

* CBOW: un par por token — `([a, b, d, e], c)`.
* Skip-gram: un par por vecino — `([c], a)`, `([c], b)`, `([c], d)`, `([c], e)`.

El contrato del batch es `(entrada, máscara, target)` en las dos, así que `train.py`
nunca ramifica por arquitectura. Se elige con una sola llave del YAML: `arch`.


In [1]:

import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.config import corpus_path, load_config, vocab_path
from src.dataset import format_pair, generate_pairs, generate_skipgram_pairs
from src.vocabulary import Vocabulary

CONFIG = load_config(ROOT / "configs" / "base.yaml")
VOCAB = Vocabulary.load(vocab_path(CONFIG))
TOK = CONFIG["tokenizer"]
print("vocabulario:", VOCAB)


vocabulario: Vocabulary(size=5000, total_tokens=2,028,209,650, coverage=0.8326, min_count=5)



## Demostración 1 — La misma ventana, leída de dos maneras

`generate_skipgram_pairs` **no** recorre el corpus por su cuenta: llama a
`generate_pairs` y expande cada par. Por eso el muestreo de líneas, el subsampling,
`drop_unknown` y el reparto entre workers son literalmente el mismo código.


In [2]:

JUGUETE = ROOT / "data" / "processed" / "_demo" / "juguete.txt"
JUGUETE.parent.mkdir(parents=True, exist_ok=True)
JUGUETE.write_text("el gato juega en la casa\n", encoding="utf-8")

C = 2
kw = dict(subsampling_threshold=0, **TOK)

print("CBOW (un par por token):")
cbow = list(generate_pairs(JUGUETE, VOCAB, C, **kw))
for ctx, tgt in cbow:
    print("  ", format_pair(ctx, tgt, VOCAB))

print("\nSkip-gram (un par por vecino):")
sg = list(generate_skipgram_pairs(JUGUETE, VOCAB, C, **kw))
for ctx, tgt in sg:
    print("  ", format_pair(ctx, tgt, VOCAB))

print(f"\npares CBOW = {len(cbow)} | pares skip-gram = {len(sg)}  ({len(sg) / len(cbow):.1f}x)")


CBOW (un par por token):
   [gato juega]  ->  el
   [el juega en]  ->  gato
   [el gato en la]  ->  juega
   [gato juega la casa]  ->  en
   [juega en casa]  ->  la
   [en la]  ->  casa

Skip-gram (un par por vecino):
   [el]  ->  gato
   [el]  ->  juega
   [gato]  ->  el
   [gato]  ->  juega
   [gato]  ->  en
   [juega]  ->  el
   [juega]  ->  gato
   [juega]  ->  en
   [juega]  ->  la
   [en]  ->  gato
   [en]  ->  juega
   [en]  ->  la
   [en]  ->  casa
   [la]  ->  juega
   [la]  ->  en
   [la]  ->  casa
   [casa]  ->  en
   [casa]  ->  la

pares CBOW = 6 | pares skip-gram = 18  (3.0x)



Los dos conjuntos salen de las mismas ventanas: cada par de skip-gram es un elemento
del contexto de un par de CBOW, con los papeles invertidos. La comprobación siguiente
lo verifica en lugar de asumirlo.


In [3]:

esperado = [([tgt], vecino) for ctx, tgt in cbow for vecino in ctx]
assert sg == esperado
print("skip-gram == expansión exacta de CBOW  ->", sg == esperado)


skip-gram == expansión exacta de CBOW  -> True



## Demostración 2 — Dos YAML gemelos: cambia `arch` y nada más

`piloto_sg_5k_50_2` es el gemelo exacto de `piloto_5k_50_2`. Ambos apuntan al mismo
vocabulario con `vocab_from`.


In [4]:

cbow_cfg = load_config(ROOT / "configs" / "piloto_5k_50_2.yaml")
sg_cfg = load_config(ROOT / "configs" / "piloto_sg_5k_50_2.yaml")

difieren = {k for k in cbow_cfg.keys() | sg_cfg.keys() if cbow_cfg.get(k) != sg_cfg.get(k)}
for k in sorted(difieren):
    print(f"{k:<12} CBOW={cbow_cfg.get(k)!r:<24} skip-gram={sg_cfg.get(k)!r}")
print("\nvocab_from:", cbow_cfg["vocab_from"], "==", sg_cfg["vocab_from"])


arch         CBOW='cbow'                   skip-gram='skipgram'
name         CBOW='piloto_5k_50_2'         skip-gram='piloto_sg_5k_50_2'

vocab_from: base_5k_50_2 == base_5k_50_2



Si sale más de `name` y `arch` en la tabla de arriba, el experimento dejó de aislar la
arquitectura y hay que revisarlo antes de interpretar resultados.

## Lo que sí cambia, y por qué se cuenta aparte

Tres cosas justifican que skip-gram tenga su propia carpeta de notebooks aunque no
tenga su propio código:

1. **La loss no se compara entre arquitecturas.** Son tareas distintas sobre conjuntos
   de pares de tamaños distintos. La comparación válida es la precisión en analogías,
   y mejor aún McNemar sobre los mismos ítems.
2. **El costo es distinto.** Una época de skip-gram cuesta ~2,8x más pares con
   `context_size=2` y ~5x con `context_size=5`.
3. **Los resultados son un relato propio:** empate a contexto 2, y a contexto 5
   skip-gram queda por debajo de CBOW.

## Mapa de carpetas

| Carpeta | Contenido |
|---|---|
| `notebooks_CBOW/` | `00_resumen_proyecto` y las fases 1-7: corpus, vocabulario, pares, entrenamiento, evaluación y comparación de configuraciones CBOW |
| `notebooks_SKIPGRAM/01_por_que_no_un_proyecto_nuevo` | este notebook |
| `notebooks_SKIPGRAM/02_skipgram` | Fase 8: skip-gram contra CBOW (empate) |
| `notebooks_SKIPGRAM/03_skipgram_contexto5` | Fase 9: arquitectura × contexto |
| `notebooks_SKIPGRAM/04_resumen_comparativo` | las siete corridas y los ocho efectos |
